In [1]:
import os
import joblib
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
folders = [Path("./low-code-localization") / "artifacts" / "04-Added_automl_model" / "models", Path("./JupyterLab") / "Benchmarking-results"]

In [3]:
models=[
    [
        Path("automl-spring-KFold-ExampleModel")  /"spring-ExampleModel-KFoldSplit-results.pkl", 
        Path("automl-spring-Random-ExampleModel") /"spring-ExampleModel-RandomSplit-results.pkl", 
        Path("automl-winter-KFold-ExampleModel")  /"winter-ExampleModel-KFoldSplit-results.pkl", 
        Path("automl-winter-Random-ExampleModel") /"winter-ExampleModel-RandomSplit-results.pkl"
    ],
    [
        Path("Results_ExampleModel-KFoldSplit-springSubset.pkl"),
        Path("Results_ExampleModel-RandomSplit-springSubset.pkl"),
        Path("Results_ExampleModel-KFoldSplit-winterSubset.pkl"),
        Path("Results_ExampleModel-RandomSplit-winterSubset.pkl")
    ]
    
]


In [4]:
column = []
for i in range(2):
    log = ""
    for model in models[i]:
        data     = joblib.load(folders[i] / model)
        report   = data["model_data"]["reports"][i] if i == 0 else data[0]

        scores   = report["scores"]
        params   = report["params"]
        fit_time = report["fit_time"]        

        log += model.stem + "\n"
        log += f"├╴rmse: {scores['rmse']['mean']: 8.6f} ± {scores['rmse']['std']: 8.6f}"+ "\n"
        log += f"├╴r_squared: {scores['r_squared']['mean']: 8.6f} ± {scores['r_squared']['std']: 8.6f}" + "\n"
        log +="├╴params:" + "\n"

        arr = [f"{name}: {value}" for name, value in params.items()]
        arr = ["│ ├╴"+_ for _ in arr[:-1]] + ["│ └╴"+arr[-1]]
        log+="\n".join(arr)+ "\n"
        log+=f"└╴fit_time: {fit_time['mean']} ± {fit_time['std']}"+ "\n"
        log+="\n"

    column.append(log)

2025-07-25 11:52:25.564386: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-07-25 11:52:25.579918: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-07-25 11:52:25.579954: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-07-25 11:52:25.591053: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-07-25 11:52:26.525682: W tensorflow/compiler/tf

In [5]:
import difflib

def side_by_side_diff(a: str, b: str, width=55):
    a_lines = a.splitlines()
    b_lines = b.splitlines()
    
    diff = list(difflib.ndiff(a_lines, b_lines))
    
    left = []
    right = []
    for line in diff:
        if line.startswith("  "):  # same
            left.append("✔")
            right.append("")
        elif line.startswith("- "):  # only in a
            # show A version
            left.append(f"❌{line[2:]:<{width}} –>")
        elif line.startswith("+ "):  # only in b
            # show B version
            right.append(f" {line[2:]}")  # append next to the previous removed line
    
    return "\n".join(left),"\n".join(right)

ld, rd = side_by_side_diff(column[0], column[1])

# for line_l, line_r in zip(column[0].splitlines(), column[1].splitlines()):
    
    

In [6]:
def two_columns(left_block, right_block, width=56, gap=4):
    # Split blocks into lines
    left_lines = left_block.splitlines()
    right_lines = right_block.splitlines()
    
    # Pad so both have the same number of lines
    max_lines = max(len(left_lines), len(right_lines))
    left_lines += [''] * (max_lines - len(left_lines))
    right_lines += [''] * (max_lines - len(right_lines))
    
    # Align each pair of lines
    merged = []
    for l, r in zip(left_lines, right_lines):
        merged.append(f"{l:<{width}}{' ' * gap}{r}")
    
    return "\n".join(merged)
    
    # Combine into side-by-side columns
    #return "\n".join(f"{l:<{width}}{' ' * gap}{r}" for l, r in zip(left_lines, right_lines))

out = two_columns(column[0], column[1])
diff = two_columns(ld, rd)
print(two_columns(out, diff, width = 115))

spring-ExampleModel-KFoldSplit-results                      Results_ExampleModel-KFoldSplit-springSubset               ❌spring-ExampleModel-KFoldSplit-results                  –>     Results_ExampleModel-KFoldSplit-springSubset
├╴rmse:  1.569384 ±  0.086069                               ├╴rmse:  1.569384 ±  0.086069                              ✔                                                           
├╴r_squared:  0.153710 ±  0.130538                          ├╴r_squared:  0.153710 ±  0.130538                         ✔                                                           
├╴params:                                                   ├╴params:                                                  ✔                                                           
│ ├╴general_block_1/dense_block_1/use_batchnorm: True       │ ├╴general_block_1/dense_block_1/use_batchnorm: True      ✔                                                           
│ ├╴general_block_1/dense_block_1/num_layers: 3     